# Fields to update
1. MB_ID (Done)
2. Country
3. Sub_genre
4. Homepage
5. Bandcamp page

In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:
%pip install -r requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: c:\Users\dsvel\.pyenv\pyenv-win\versions\3.12.9\python.exe -m pip install --upgrade pip


In [3]:
import sys
print(sys.version)
print(sys.executable)

3.12.9 (tags/v3.12.9:fdb8142, Feb  4 2025, 15:27:58) [MSC v.1942 64 bit (AMD64)]
c:\Users\dsvel\.pyenv\pyenv-win\versions\3.12.9\python.exe


In [4]:
# Notion imports
from Notion.reader import get_artists_db
from Notion.writer import build_property, update_artist, heading_block, paragraph_block, bullet_block
from Notion.database import myNotionDatabases
from Data.artists import Artist
from Data.tags import clean_tags
from Datasources.Last_Fm import get_artist_tags
# MusicBrainz imports
from Datasources.MusicBrainz import search_artist,get_artist_info
from Datasources.Wikipedia import get_wikipedia_data
from Datasources.llm_client import llm_page_entry
from tqdm import tqdm
import random
import copy

In [5]:
notion_response = get_artists_db()
#artists_db = []
db = myNotionDatabases()
Updated_notion_artist_database = {}

for artist in notion_response:
    extracted_artist = db.extract_artist(artist)
    #artists_db.append(extracted_artist)
    key = extracted_artist['artist_name'].lower().strip()
    Updated_notion_artist_database[key] = Artist(
        name=extracted_artist['artist_name'],
        mbid=extracted_artist['mb_id'],
        country=extracted_artist['country'],
        tags=extracted_artist.get('subgenre', []),
        website_url=extracted_artist.get('website'),
        bandcamp_url=extracted_artist.get('bandcamp_url'),
        wikidata_url=extracted_artist.get('wikidata_url'),
        bandsintown_url=extracted_artist.get('bandsintown_url'),
        page_id=extracted_artist.get('id'),
        page_data=""
    )


Original_notion_artist_database = copy.deepcopy(Updated_notion_artist_database)

In [6]:
#for artist in random.sample(sorted(notion_artist_database.values(), key=lambda a: a.name), min(10, len(notion_artist_database))): #Selecting X random artists for testing purposes
no_matches = []
for artist in tqdm(Updated_notion_artist_database.values()):
    if artist.mbid is None:
        try:
            mb_id, match_result = search_artist(artist_name=artist.name.strip(), confidence_threshold=85)
            if mb_id is None:
                #this will force an update and send the result to the notion database. 
                # I can then make a view which filters for them.
                artist.mbid = f"No match found for |{artist.name}| with the reason: {match_result}"
            else:
                artist.mbid = mb_id
        except Exception as e:
            print(f"Error processing {artist.name}: {e}")

    if artist.mbid is not None and not artist.mbid.startswith("No match found"):
        try:
            mb_artist_info = get_artist_info(artist.mbid)
            if artist.country is None:
                artist.country = mb_artist_info.get("country")
            if artist.website_url is None:
                artist.website_url = mb_artist_info.get("homepage")
            if artist.bandcamp_url is None:
                artist.bandcamp_url = mb_artist_info.get("bandcamp")
            if artist.wikidata_url is None:
                artist.wikidata_url = mb_artist_info.get("wikidata")
            if artist.bandsintown_url is None:
                artist.bandsintown_url = mb_artist_info.get("Bandsintown")
            if not artist.tags:
                artist.tags = mb_artist_info.get("tags", [])
        except Exception as e:
            print(f"Error fetching info for {artist.name}: {e}")


100%|██████████| 144/144 [00:00<00:00, 95808.97it/s]


In [7]:
for error in no_matches:
    print(error)
print(f"Number of missing matches = {len(no_matches)}")
no_exact_match = len([x for x in no_matches if "No Exact Match" in x])
print(f"No Exact Match Issues = {no_exact_match}")
multi_exact_match = len([x for x in no_matches if "Multiple Exact Matches" in x])
print(f"Multiple Exact Match Issues = {multi_exact_match}")

Number of missing matches = 0
No Exact Match Issues = 0
Multiple Exact Match Issues = 0


In [8]:
#Get the Last.fm info for the artist.
last_fm_artists = []
for artist in Updated_notion_artist_database.values():
    if artist.mbid is not None:
        last_fm_artists.append(artist)

In [9]:
for artist in tqdm(last_fm_artists):
    #print(artist.name)
    last_fm_response = get_artist_tags(mb_id=artist.mbid)
    artist.add_tags(last_fm_response)

100%|██████████| 144/144 [00:23<00:00,  6.20it/s]


In [10]:
for artist in tqdm(last_fm_artists):
    cleaned_tags = clean_tags(artist.tags)
    sorted_tags = sorted(cleaned_tags.items(), key=lambda x: x[1], reverse=True)
    Top_x = sorted_tags[:3]
    declared_subgenres = []

    for tag in Top_x:
        declared_subgenres.append(tag)
    artist.tags = Top_x

100%|██████████| 144/144 [00:00<00:00, 143873.22it/s]


In [11]:
def build_artist_page_blocks(llm_result: dict) -> list:
    """Builds a list of Notion blocks for an artist page based on LLM results."""
    blocks = []

    if llm_result.get("overview"):
        blocks.append(heading_block("Overview"))
        blocks.append(paragraph_block(llm_result["overview"]))

    if llm_result.get("sound"):
        blocks.append(heading_block("Sound"))
        blocks.append(paragraph_block(llm_result["sound"]))

    if llm_result.get("activity"):
        blocks.append(heading_block("Activity"))
        blocks.append(paragraph_block(llm_result["activity"]))

    discography = llm_result.get("discography", {})
    if discography:
        blocks.append(heading_block("Discography"))
        for release_type, count in discography.items():
            blocks.append(bullet_block(f"{release_type}: {count}"))

    return blocks

In [12]:
for current_artist in Updated_notion_artist_database.values():
    if current_artist.name.lower() != "powerwolf":  # TEMP: testing only
        continue
    if current_artist.wikidata_url is not None:
        wikidata_id = current_artist.wikidata_url.split("/")[-1]
        summary, discography = get_wikipedia_data(wikidata_id)
        if summary is None:
            continue
        llm_data = "Summary:\n" + summary + "\n\n" + "Discography:\n" + discography
        try:
            llm_result = llm_page_entry(llm_data)
        except ValueError:
            print(f"Could not generate page data for {current_artist.name} — skipping.")
            continue
        current_artist.page_data = build_artist_page_blocks(llm_result)
        if llm_result.get("sub_genres") and current_artist.tags is None:
            current_artist.tags = []
            current_artist.tags.extend([(genre, None) for genre in llm_result["sub_genres"]])

Artist has wiki page and discography page.
Attempt 1 failed to parse LLM response. Retrying...


In [ ]:
for key, original in Original_notion_artist_database.items():
    if key != "powerwolf": # TEMP: testing only
        continue
    updated_data = Updated_notion_artist_database[key]
    updates = {}
    page_id = original.page_id
    if 'No match found' in updated_data.mbid:
        updates["MB_ID"] = ("rich_text", updated_data.mbid)
    else:
        if original.mbid is None and updated_data.mbid != None:
            updates["MB_ID"] = ("rich_text", updated_data.mbid)
        if original.country is None and updated_data.country != None:
            updates["Country"] = ("rich_text", updated_data.country)
        if original.bandcamp_url is None and updated_data.bandcamp_url != None:
            updates["Bandcamp url"] = ("url", updated_data.bandcamp_url)
        if original.website_url is None and updated_data.website_url != None:
            updates["Website"] = ("url", updated_data.website_url)
        if original.wikidata_url is None and updated_data.wikidata_url != None:
            updates["Wikidata_URL"] = ("url", updated_data.wikidata_url)
        if original.bandsintown_url is None and updated_data.bandsintown_url != None:
            updates["Bandsintown_URL"] = ("url", updated_data.bandsintown_url)
        if original.tags is None and updated_data.tags != []:
            updates["Subgenre"] = ("multi_select", [tag[0] for tag in updated_data.tags])
    
    if len(updates) != 0:
        update_artist(page_id=page_id, updates=updates, dry_run=True)#This line acts as a printout
        update_artist(page_id=page_id, updates=updates, dry_run=False, body_blocks=updated_data.page_data)

DRY RUN — 3396e950-cac6-8043-a5ec-d36a23b9a6f0:
  Subgenre: {'multi_select': [{'name': 'power metal'}, {'name': 'heavy metal'}, {'name': 'german'}]}
